
##Transform Results Data
 1. Read bronze `results` table
 1. Keep only the columns required for analytics (Drop `url` column)
 1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
 1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
 1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
 1. Remove duplicate records
 1. Transform values of column `race_name` to Title Case
 1. Write the transformed data to silver `results` table

##Entity Relationship Diagram- Formula1 Bronze Schema
![formula1-silver-data-erd.png](./formula1-silver-data-erd.png "formula1-silver-data-erd.png")

In [0]:
%run ../00-common/1.Environment-Config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"


## Step 1 Read bronze results table

In [0]:
results_df = spark.read.table(bronze_table)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
results_selected_df = (
    results_df.select("season",
                     "round",
                     "constructorId",
                     "driverId",
                     "date",
                     "raceName",
                     "grid",
                     "laps",
                     "number",
                     "points",
                     "position",
                     "positionText",
                     "status",
                     "ingestion_timestamp",
                     "source_file"
                      )
)

In [0]:
display(results_selected_df)

##Step 3 & 4 - Standardise Column Names
 - Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
 - Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)

In [0]:
results_renamed_df = (
    results_selected_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text"
        })
)

In [0]:
display(results_renamed_df)

### Step 5 - Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)


In [0]:
from pyspark.sql import functions as F

In [0]:
results_valid_df = (
    results_renamed_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
)

In [0]:
display(results_valid_df)

In [0]:
display(results_renamed_df.count()-results_valid_df.count())

#### Step 5 - Remove duplicate records

In [0]:
results_distinct_df= results_valid_df.dropDuplicates(["season","round","constructor_id","driver_id"])



In [0]:
display(results_valid_df.count() - results_distinct_df.count())

####Step 7 - Transform values of column `race_name` to Title Case

In [0]:
results_final_df = (
    results_distinct_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

In [0]:
display(results_final_df)

####Step 8 - Write the transformed data to silver `results` table

In [0]:
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))